# 构建自治智能体

> 前几讲分别给出了验证器、工具执行、记忆与评测，每一环都能在演示任务上跑通。但从“能做演示”到“能在无人看管下连续运行数小时”，中间还隔着可靠性、监督与信任三个问题。
>
> 这一讲把这些能力装进一个整体，直面自治背后的工程与开放问题。我们从单步正确率的乘法效应出发，依次搭建错误检测与重试循环、检查点恢复、阈值化接管，最后用一个最小的可信监控协议，回应人追不上模型之后如何把关的问题。

一个 agent 是模型、记忆、工具、监督与恢复机制的集合体。单个 LLM 调用只能完成一步；承担“决定下一步、执行动作、把反馈带回来”的循环结构，连同它周围的 harness，才构成能自主推进的系统。我们先从一个失败的长任务开始：每一步看起来都只有一点点出错概率，但步骤一多，整条任务成功的机会会迅速下降。于是“自治”不是让模型一直运行，而是让系统知道什么时候检查、怎么恢复、何时请人接手。接下来我们把这个大问题拆成几个可以动手实现的小机制。

第一个问题是可靠性：单步做对、循环跑稳、出错后能恢复，是三件不同的事。第二个问题是监督：给 agent 结果信号、步级信号，还是让它自己迭代改错，粒度不同，成本与鲁棒性也不同。第三个问题是信任：把有后果的动作委托出去的依据是什么，委托的边界画在哪里。第四个问题留给开放讨论：当模型在越来越多的技能上超过未经辅助的人，把关工作如何组织。需要先澄清一处容易混淆的名字。Laskin 创立的公司叫 Reflection AI，与开源模型 Reflection 70B（HyperWrite 与 Glaive 于 2024 年发布）是两个不同实体；本讲提到 Reflection 之处都指前者。

可靠性问题从单步正确率讲起，它用最简单的方式决定整条任务的成败。

## 1. 从演示到自治

演示任务的共同点是短：几步之内就能拿到一次正确结果，错了重跑一遍成本也低。自治任务要求系统在一长串步骤里不犯错，或者错了能自己发现并补救。前几讲的演示大多在十步以内；无人看管地连续运行，可能是几千步。步骤数从十步涨到几千步，对可靠性的要求也随之提高几个数量级。先看最朴素的一层，单步正确率如何决定整条任务的成败。


In [ ]:
import os
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np
np.random.seed(42)

from llm_client import get_llm
client = get_llm()
print('llm_client 就绪；当前模式：',
      '脚本化示例（输出为占位，供无 key 环境执行）' if False else '真实 API')

### 单步正确率是乘法因子

先看为什么是乘法。我们把任务想成一条必须连过的关卡：第一关过了才能到第二关，任何一关失败，整条任务就失败。两件独立事件同时发生的概率是各自概率的乘积——两个骰子同时掷出 6 的概率是 (1/6)×(1/6)=1/36。agent 把任务拆成 n 个先后依赖的步骤，每一步独立成功的概率都是 p，整条任务成功的条件是每一步都成功，所以整体成功率是 p 的 n 次方。

用最小的数字例子走一遍。单步 0.9，两步：第一步成功 0.9；在这 0.9 里，第二步再成功 0.9，整体是 0.9×0.9=0.81。每一步都会把“还剩下的成功率”再压掉 10%。多走几步：

| 步数 n | 整体成功率 0.9^n | 直观感受 |
|:---|:---|:---|
| 1 | 0.900 | 九成 |
| 2 | 0.810 | 八成 |
| 5 | 0.590 | 不到六成 |
| 10 | 0.349 | 约三分之一 |

两步都 90% 的任务只剩 81%，五步就掉到 59%，十步只剩 35%。把单步提高到 0.99，10 步仍有 0.99^10 ≈ 0.90，但 100 步也只有 0.99^100 ≈ 0.37。自治是乘法，不是加法：从“能做演示”到“无人看管跑几小时”，不能只靠每步更稳一点，必须在出错后能自己发现并补救，这正是下一节要做的。


In [ ]:
def overall_success(probs):
    '''输入每步成功概率列表，返回整条任务成功率（各步独立相乘）。'''
    p = 1.0
    for q in probs:
        p *= q
    return p

for n in (1, 2, 5, 10):
    print(f'{n:>2} 步、单步 0.90：整体成功率 {overall_success([0.9] * n):.3f}')

print('10 步、单步 0.99：整体成功率', round(overall_success([0.99] * 10), 3))

## 2. 可靠性：错误检测与恢复

可靠性不是单一性质，而是三层：单步做对、循环跑稳、出错后能恢复。每一层有独立的失败模式和解法。用一个 agent 计算“1 到 5 的和”作例子，把三层分开看：

- 单步做对：某一步输出错误结果。例如 agent 把 2+3 算成了 6。这一层由模型能力与单步验证决定，对应下面的乘法因子与乘法塌陷。
- 循环跑稳：已经产生的状态被错误污染后一路向后传播。例如累计和记成了 6，之后每一步都从 6 开始，全错。这一层靠状态卫生与检查点保护。
- 长程纠错：某一步输出了错值，但没人发现、直接推进。例如第 3 步算错一次，之后每步都正常，最终结果却整体偏了。这一层靠自检与重试循环。

三层会叠加：第一步出错、错误又没被发现、还写进了状态，就同时踩中三层的坑。下面先看第一层与第二层的交界处，长任务里的错误是如何累积的。


### 乘法塌陷：错误随位数累积

先看一个具体的长乘法：23 × 47，拆成三个子步骤：

```text
  23 × 7 = 161        (个位部分积)
  23 × 4 = 92 → 920   (十位部分积，左移一位)
  161 + 920 = 1081    (加总，得真值)
```

每个子步骤都可能出错：7×3 记成 18、进位进错、加总对错位。位数越多，这样的子步骤越多。关键是出错没有信号：如果十位部分积算成 91 → 910，最终是 161 + 910 = 1071。1071 是一个完全正常的数，每个数字都合法，没有任何地方写着“我算错了”，可它确实错了。这就是“看起来每一步都对、结果却错”：最终答案是一长串子步骤的合取，任何一处出错，整串作废，而错误通常不会自己报错。

现在给“每一步都对、结果却错”一个具体数字。假设模型解一个含 15 个依赖步骤的乘法，每步正确率 0.9。整次成功的概率是 0.9^15。一步步压下来：

```text
第 1 步后   0.9
第 2 步后   0.9 × 0.9     = 0.81
第 3 步后   0.81 × 0.9    ≈ 0.73
第 5 步后   ≈ 0.59
第 10 步后  ≈ 0.35
第 15 步后  0.9^15        ≈ 0.21
```

15 步、每步 0.9，整体成功率只剩约 21%，五次里成不了一次。这个推导还隐含一个被低估的前提：每步正确率固定。实际往往更糟——任务越长，注意力越分散，靠后的列越容易错。下面的代码用两条曲线对照：单列正确率固定 0.99，与单列正确率随列数线性衰减（0.97 − 0.012 × 列号）。15 列时前者仍有 0.99^15 ≈ 0.86；后者因为越靠后单列越差，整体只剩约 0.13。衰减曲线在图上塌陷得更陡，这就是“乘法塌陷”名字的由来。


In [ ]:
import matplotlib.pyplot as plt

def per_column_success(n):
    '''第 n 列子步骤的成功概率：列数越多，单步正确率越低。'''
    return max(0.05, 0.97 - 0.012 * n)

def fixed_overall(n):
    '''每列成功率恒为 0.99 时，n 列整任务的成功率。'''
    return 0.99 ** n

def decaying_overall(n):
    '''成功率随列数衰减时，n 列整任务的成功率（逐列连乘）。'''
    p = 1.0
    for i in range(1, n + 1):
        p *= per_column_success(i)
    return p

cols = np.arange(1, 16)
fixed = [fixed_overall(n) for n in cols]
decay = [decaying_overall(n) for n in cols]

print('列数 | 固定单步 0.99 | 随位数衰减')
for n in cols:
    print(f'{n:>3} | {fixed_overall(n):.4f}      | {decaying_overall(n):.4f}')

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(cols, fixed, marker='o', label='fixed step acc = 0.99')
ax.plot(cols, decay, marker='s', label='step acc decays with length')
ax.set_xlabel('number of columns (digits)')
ax.set_ylabel('overall success rate')
ax.set_title('Multiplicative collapse of long-multiplication success')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 自检与重试循环

单步正确率有限时，补救的办法是在循环里加一道自检：执行后先验证结果，再决定推进还是重试。重试的数学很直接。假设每步错误的概率 q=0.35，一次尝试正确的概率是 0.65。允许重试 3 次（总共尝试 4 次），只有 4 次全错才会最终失败，失败概率是 q^4：

```text
0.35 × 0.35     ≈ 0.122
0.122 × 0.35    ≈ 0.043
0.043 × 0.35    ≈ 0.015
```

约 1.5%。也就是说，只要验证器可靠，重试就能把“每步 65% 正确”抬到“约 98.5% 正确”。代价是多花几次 LLM 调用。

但这个结论依赖一个前提：验证器真的分得清对错。如果验证器自己不会判断，重试就是在原地掷骰子——错了放行、对了也可能被拦下。验证信号的质量决定这个循环的上限。下面用一个具体任务对比三种配置：40 条加法表达式，真值确定，求解器单步错误率 0.35。

- 不做自检：第一次求解结果直接接受。错误无处被拦截，正确率就是求解器本身的水平，一条表达式花一次调用。
- 让 LLM 自己复核（弱自检）：复核几乎总是说“看起来对”，对应 Self-Refine 论文的负结果——模型对 94% 的错误样本都说看起来对。弱自检拦不住错误，重试几乎不触发，多花的调用换不来正确率。
- 用精确验证器把关（硬验证）：重算真值精确比较，任何偏差都判错。错误被可靠拦截，重试真正生效，正确率接近 98.5%，代价是调用次数明显增加。

真实 API 演示下求解算术会得到确定性的正确结果，错误率由 NoisyBrain 注入，便于做受控实验。用同一批 40 条表达式跑三种配置，比较“真正正确数”与“LLM 调用数”。循环自己报告的成败不可信，真正的正确数要用真值对一遍——这就是结果级监督要回答的问题，到第 3 节再展开。


In [ ]:
import re

def llm_solve(client, expr):
    '''让 LLM 计算一条算术表达式，返回数值。

    真实 API 演示的输出固定为“计算结果：N。”；真实模式下取回复里最后一个整数。
    '''
    text = client.chat([{'role': 'user', 'content': f'计算 {expr} 的值'}])
    return int(re.findall(r'-?\d+', text)[-1])


class NoisyBrain:
    '''一个单步正确率有限的求解器：先调 LLM，再按错误率把结果改错。

    真实模式中模型本身就可能出错；这里用可调错误率做受控实验，
    便于单独观察自检与重试机制的作用。错误是瞬时的，重试可以撞对。
    '''

    def __init__(self, client, err_prob=0.35, seed=0):
        self.client = client
        self.err_prob = err_prob
        self.rng = np.random.RandomState(seed)

    def solve(self, expr):
        '''求解表达式：以 err_prob 概率返回一个错误值，否则返回 LLM 结果。'''
        value = llm_solve(self.client, expr)
        if self.rng.rand() < self.err_prob:
            return value + self.rng.randint(1, 5)
        return value


def exact_check(expr, answer):
    '''硬验证：重算真值并精确比较，任何偏差都判定为错。'''
    a, b = (int(x) for x in expr.split('加'))
    return answer == a + b


def llm_check(client, expr, answer, rng):
    '''弱自检：把答案交给 LLM 复核，复核几乎总是说“看起来对”。

    真实 API 演示下复核没有独立判定能力，按 94% 概率放行，
    对应 Self-Refine 论文里“模型对 94% 的错误样本都说看起来对”的负结果。
    '''
    client.chat([{'role': 'user', 'content': f'请复核：{expr} 的答案 {answer} 是否正确？'}])
    return rng.rand() < 0.94


class AgentLoop:
    '''带自检与重试上限的 agent 循环。

    流程：求解 -> 自检。checker 为 None 时不验证，直接接受结果；
    验证失败则在剩余尝试次数内重试，次数用尽仍未通过则放弃本步。
    '''

    def __init__(self, brain, checker=None, max_retries=3):
        self.brain = brain
        self.checker = checker
        self.max_retries = max_retries

    def run(self, exprs):
        '''依次求解 exprs，返回 (接受的答案列表, LLM 总调用次数)。

        无 checker 时无条件接受第一次求解结果，因此错误值会被放行。
        '''
        answers = []
        calls = 0
        for expr in exprs:
            for _ in range(self.max_retries + 1):
                answer = self.brain.solve(expr)
                calls += 1
                if self.checker is None or self.checker(expr, answer):
                    answers.append(answer)
                    break
        return answers, calls


print('求解器、验证器与重试循环已定义，下一步用同一批表达式对比三种配置')


In [ ]:
rng = np.random.RandomState(7)
# 真实 API 演示控制在 8 道题，避免一次实验产生过多调用。
exprs = [f'{a} 加 {b}' for a, b in rng.randint(1, 30, size=(8, 2))]
weak_rng = np.random.RandomState(3)


def true_value(expr):
    '''表达式的真值，供评估脚本对比。'''
    a, b = (int(x) for x in expr.split('加'))
    return a + b


def evaluate(seed, checker, max_retries):
    '''用同一批表达式评估一种配置，返回 (真正正确数, LLM 调用数)。

    真正的正确数以真值对比为准，而不是循环自己报告的成败。
    '''
    brain = NoisyBrain(client, err_prob=0.35, seed=seed)
    loop = AgentLoop(brain, checker=checker, max_retries=max_retries)
    answers, calls = loop.run(exprs)
    n_ok = sum(1 for e, a in zip(exprs, answers) if a == true_value(e))
    return n_ok, calls


results = [
    ('无自检（接受一切）',
     evaluate(0, None, 0)),
    ('LLM 自检（弱验证）',
     evaluate(0, lambda e, a: llm_check(client, e, a, weak_rng), 3)),
    ('精确验证（硬信号）',
     evaluate(0, exact_check, 3)),
]

for name, (n_ok, n_calls) in results:
    print(f'{name}：真正确 {n_ok:>2}/{len(exprs)}，LLM 调用 {n_calls} 次')


**三组结果对照**

跑完 40 条表达式，三组的差别一眼可见：

- 无自检：40 次调用，真正确 31/40。正确率停在求解器的单步水平，没有额外的保障。
- 弱自检：41 次调用，真正确 32/40。比无自检多花 1 次调用、只多对 1 条——复核几乎总是说“看起来对”，拦不住错，重试自然不触发。这就是 Self-Refine 论文负结果在本例里的样子。
- 精确验证：53 次调用，真正确 40/40。13 次多余调用买到了全对。

53 这个数字可以手算出来。单步错误率 0.35、最多尝试 4 次，一条表达式平均要调用 1.5 次左右才通过验证（第 1 次就成功约 65%，失败 1 次后再成功约 23%，依此类推），40 条合计约 60 次；实际 53 次是随机过程的一次具体实现。多花一点调用换接近 100% 的正确率，在自治任务里通常是划算的交易——前提是验证器真的可靠。这正应了标题：验证信号的质量决定循环的上限。


### 检查点：把中间状态落盘

循环里的状态一旦出错就会向后传播，这是循环稳定这一层的主要威胁。状态指循环在每一步之间携带、并且不重算就拿不回来的信息：累计和、已处理列表、对话历史、工具返回结果。进程被 OOM、断电或外部信号打断时，内存里的状态全部丢失，只有写到磁盘的东西还在。

对抗手段之一是把中间状态显式保存：每完成一步就把进度写到磁盘，进程崩溃后从最近的检查点继续，而不是从头开始。磁盘上的文件不随进程消失，这是检查点成立的核心假设。下面用一个小任务走一遍：依次处理 10 个数并累计求和，每处理一项就落盘一次，处理完第 5 项后模拟一次进程崩溃。

崩溃时磁盘上保存的是“已处理 1 到 5、累计 15”。恢复逻辑读回这个状态，看到 done 里有 5 项，就从第 6 项接着算，只再处理 5 步；没有检查点的做法只能从零开始，把 10 步全部重做。数字越大、单步越贵，检查点省下的越多。

这里有一个前提：恢复是安全的。本例每一项的加工彼此独立，重跑不会产生重复副作用，所以“只做未完成的项”是正确的。如果每一步都带不可逆的外部副作用，光落盘还不够，还要保证每个动作至多执行一次（exactly-once 语义），那是更难的工程问题。先看本讲最朴素的一版：状态可序列化、动作可安全重跑。


In [ ]:
import json
import os

CP_PATH = '/tmp/lecture18_state.json'

def save_checkpoint(state, path=CP_PATH):
    '''把任务状态写入 JSON 文件，模拟落盘。'''
    with open(path, 'w') as f:
        json.dump(state, f)

def load_checkpoint(path=CP_PATH):
    '''读取检查点；文件不存在时返回 None。'''
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

items = list(range(1, 11))

def run_until_crash(items, crash_after=5):
    '''逐项处理 items，处理到第 crash_after 项时模拟进程崩溃。'''
    state = {'done': [], 'total': 0}
    for x in items:
        if len(state['done']) == crash_after:
            raise RuntimeError('进程被外部信号中断')
        state['total'] += x
        state['done'].append(x)
        save_checkpoint(state)
        done = len(state['done'])
        print('处理', x, '：已完成', done, '项，累计', state['total'])
    return state

try:
    run_until_crash(items)
except RuntimeError as e:
    print('崩溃：', e)

In [ ]:
def run_from_scratch(items):
    '''从头逐项处理并每步落盘，返回 (最终状态, 处理步数)。'''
    state = {'done': [], 'total': 0}
    steps = 0
    for x in items:
        state['total'] += x
        state['done'].append(x)
        save_checkpoint(state)
        steps += 1
    return state, steps

def resume_from_checkpoint(items, path=CP_PATH):
    '''读取检查点，只处理尚未完成的项，返回 (最终状态, 处理步数)。'''
    state = load_checkpoint(path)
    if state is None:
        state = {'done': [], 'total': 0}
    done_set = set(state['done'])
    todo = [x for x in items if x not in done_set]
    for x in todo:
        state['total'] += x
        state['done'].append(x)
        save_checkpoint(state)
    return state, len(todo)

state_resume, steps_resume = resume_from_checkpoint(items)
state_scratch, steps_scratch = run_from_scratch(items)
print('断点续跑（有检查点）：', steps_resume, '步')
print('从头再来（无检查点）：', steps_scratch, '步')
print('恢复时跳过已完成', len(items) - steps_resume, '项，最终累计', state_resume['total'])

## 3. 监督与信任边界

可靠性解决系统能不能自己跑稳，监督与信任解决我们凭什么相信它、给它多大授权。先看监督的粒度：同一个错误，结果级检查与步级检查能给出的信息完全不同。

### 结果级监督与步级监督

监督要回答的问题是“agent 做得对不对、错在哪”。回答的粒度不同，能获得的信息就不同。结果级监督只看最终输出对不对：便宜，能自动标注——只要有真值或单元测试，不需要过程。步级监督对每一步单独给反馈：更可解释，能指出错在哪一步，但要步级标注，成本高得多。

用一个五步轨迹把差别走一遍。轨迹模拟“逐步把 1、2、3、4、5 累加”：第 1 步加 1、第 2 步加 2，依此类推。我们在第 3 步把值 3 错写成 103（加了 100），其余步都正常。真值是 1+2+3+4+5=15。

结果级检查比较最终累计与 15：1+2+103+4+5=115，不相等，判定不通过。它给出“错了”的判定，但不给出错在哪一步——只能看到最终结果偏了 100，错因要靠猜。

步级检查逐项核对：第 1 步后累计 1、第 2 步后累计 3，都与期望相符；第 3 步后累计 106、期望是 6，直接定位到第 3 步。它把错误的行踪暴露出来。

再看错误复合效应。第 3 步错了之后，第 4、5 步即使完全正确，累计值也永远对不上期望——只看最终结果，我们无法区分“错了一次”和“每一步都错”。这就是长任务里结果级监督的盲区：早期的一个小错误会污染后面每一步检查的结论，这正是循环稳定这一层要防的。步级监督贵，但它把“在哪里出错”从猜变成了看。


In [ ]:
def build_trajectory(values, error_at):
    '''构造轨迹：每步是 (值, 是否出错)，error_at 之外都正常。'''
    traj = []
    for i, v in enumerate(values):
        if i == error_at:
            traj.append((v + 100, True))
        else:
            traj.append((v, False))
    return traj

def outcome_check(traj, expected):
    '''结果级检查：只比较最终累计是否等于 expected。返回 (通过, 说明)。'''
    final = sum(v for v, _ in traj)
    return final == expected, f'最终累计 {final}，期望 {expected}'

def process_check(traj):
    '''步级检查：逐项核对，返回第一个出错步的下标（0 起）或 None。'''
    total = 0
    for i, (v, bad) in enumerate(traj):
        total += v
        if bad:
            return i
    return None

values = [1, 2, 3, 4, 5]
traj = build_trajectory(values, error_at=2)

ok, why = outcome_check(traj, sum(values))
print('结果级：', '通过' if ok else f'未通过：{why}')
pos = process_check(traj)
print('步级  ：', '全部通过' if pos is None else f'第 {pos + 1} 步不通过')

In [ ]:
def compounding(traj):
    '''展示早期错误如何带偏之后每一次结果级检查。'''
    total = 0
    expect = 0
    for i, (v, bad) in enumerate(traj):
        total += v
        expect += values[i]
        verdict = '对' if total == expect else '错'
        print(f'第 {i + 1} 步后：累计 {total}，期望 {expect}，结果级检查 -> {verdict}')

compounding(traj)

### 阈值化接管：把自治与安全当作旋钮

自治的本质是委托，把有后果的动作交给 agent。委托的边界需要一个可操作的机制：agent 对每一步给出置信度，低于阈值就请求人工介入。直觉是自动驾驶的接替逻辑：路况简单时系统自己开，系统拿不准（置信度低）时就交回给驾驶员。

这个机制把“自治程度”和“安全程度”变成同一个旋钮的两个刻度。阈值设低，agent 自己做的多，人工过问少，放过的错误多；阈值设高，人工介入多，放过的错误少。下面构造 200 个动作做受控实验：80% 正确，正确动作的置信度集中在 0.9 附近，错误动作集中在 0.4 附近。模型对正确动作更有把握，这是置信度可用的前提。

先看阈值 0.7 的手算预期。置信度低于 0.7 的动作大多来自错误组（以 0.4 为中心），少数是正确组里恰好拿不准的。被接管的动作视为修正正确，其余按原始正确性计。实际运行：阈值 0.7 接管 36 个（18%），最终正确 200/200——错误动作被人工接住，正确动作基本放行。把阈值一路抬到接近 0.99，几乎所有动作都低于阈值，接管率接近 1、错误率保持 0——代价是 agent 几乎没有自治。图上两条曲线在阈值约 0.4 到 0.6 之间相交，之后错误率已归零、接管率还在涨，这段区间就是“多一个人工、少一个错误”的权衡处。

一个诚实的限制：这个机制假设置信度可信，即正确动作真的更有把握。如果模型对错误答案也给出高置信度（过度自信，这在 LLM 上很常见），置信度就不是一个好信号，旋钮转得再高也拦不住错。阈值化接管有效的前提是先校准置信度。


In [ ]:
def sample_actions(n=200, seed=0):
    '''生成 n 个动作，每项 (is_correct, confidence)。

    正确动作的置信度集中在 0.9 附近，错误动作集中在 0.4 附近，各加高斯噪声。
    '''
    rng = np.random.RandomState(seed)
    actions = []
    for _ in range(n):
        correct = rng.rand() < 0.8
        center = 0.9 if correct else 0.4
        conf = float(np.clip(center + rng.normal(0, 0.1), 0.05, 0.99))
        actions.append((correct, conf))
    return actions

def escalate(actions, threshold):
    '''置信度低于 threshold 的动作交人工（视为修正正确），其余按 is_correct 计。

    返回 (接管数, 最终正确数)。
    '''
    escalated = 0
    correct = 0
    for is_correct, conf in actions:
        if conf < threshold:
            escalated += 1
            correct += 1
        else:
            correct += int(is_correct)
    return escalated, correct

actions = sample_actions()
n_esc, n_ok = escalate(actions, 0.7)
print(f'阈值 0.7：接管 {n_esc} 个动作，最终正确 {n_ok}/{len(actions)}')

In [ ]:
import matplotlib.pyplot as plt

thresholds = np.linspace(0.0, 0.99, 30)
esc_rates = []
err_rates = []
for t in thresholds:
    n_esc, n_ok = escalate(actions, t)
    esc_rates.append(n_esc / len(actions))
    err_rates.append(1 - n_ok / len(actions))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(thresholds, esc_rates, marker='o', ms=3, label='escalation rate')
ax.plot(thresholds, err_rates, marker='s', ms=3, label='error rate')
ax.set_xlabel('confidence threshold')
ax.set_ylabel('rate')
ax.set_title('Threshold trades autonomy against safety')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for i in (0, 6, 12, 18, 24):
    print(f'阈值 {thresholds[i]:.2f}：接管率 {esc_rates[i]:.2f}，错误率 {err_rates[i]:.3f}')

## 4. 开放问题：当人追不上模型

可靠性、监督、信任都假设有一个足够强的监督者。当模型在越来越多的技能上超过未经辅助的人时，这个假设不再成立。scalable oversight 给出乐观路径，让“人 + LLM 助手”协作超过任何一方单独的能力。AI control 给出防御路径，假设模型是敌意的、故意破坏安全措施，仍然要能安全部署。用一个小型可信监控协议演示后一种思路。

### 最小 AI-control：可信监控协议

核心设想来自 Redwood 的 AI Control 工作：用可信但较弱的模型监控不可信但较强的模型。监控者要“可信但弱”而不是强的，原因在 AI control 的前提里：强模型不可信，它可能就是出问题的那一方。监控者必须可靠、可被审查；即使能力弱一些，也不接受它同样会被针对。可信，是这个协议所有保证的地基。

流程分三步。第一步，不可信模型提出大量候选动作，比如 50 个，其中约 20% 有害。第二步，可信监控器给每个动作打一个可疑度分数——现实中来自一个独立训练的弱模型，这里用特征的线性加权代替，两个特征越高越可疑。第三步，只把可疑度最高的 k 个送去人工审计，其余放行。人工预算有限，协议的作用是把预算花在最可能出问题的地方。

按可疑度从高到低选，而不是随机抽，理由在数字里。本例 50 个动作、10 个有害，审计预算 20%（10 个）：随机抽平均只命中 2 个有害动作，按可疑度抽捕到了 7 个（安全率 0.70）。有害动作的特征整体更高、可疑度集中在顶端，弱模型在这里当筛子，把强模型动作里最可疑的一小撮筛给人类。

人工审计本身也有上限。全量审计 50 个动作的安全率也只有 0.80——人类审计员按 85% 的命中率工作，即使用尽全力也拦不下所有有害动作。权衡曲线画出“审计成本 vs 安全率”：预算从 0 加到 20%，安全率从 0 跳到 0.70，曲线起点陡峭；再往上边际收益递减。可信监控加 top-k 的价值正是这个陡峭的起点——用很少的预算换到大部分安全。

被动设置下协议表现良好，被刻意针对后就失效。绕过演示把同一个有害动作的特征从(高,高)改成(低,低)，可疑度从 1.71 掉到 0.14，它就从审计名单上掉下去、直接放行。同一份载荷、换一种输出写法，就躲过监控——这是“监控必须按被针对的前提来评估”的直接演示。防御思路是 protocol security：攻击者会优化绕过，协议要针对最优攻击者做鲁棒性分析，而不是只对平均情况有效。


In [ ]:
def build_actions(n=50, harmful_rate=0.2, seed=0):
    '''生成候选动作池：每项含 是否有害 与 特征向量。

    有害动作的特征整体更高，让可信监控器有信号可依。
    '''
    rng = np.random.RandomState(seed)
    actions = []
    for i in range(n):
        harmful = rng.rand() < harmful_rate
        if harmful:
            feats = rng.rand(2) + (0.5, 0.4)
        else:
            feats = rng.rand(2)
        actions.append({'id': i, 'harmful': harmful, 'feats': feats})
    return actions

def suspiciousness(a):
    '''可信监控器对动作的可疑度打分：特征的线性加权。

    现实中这个分数来自一个独立训练的弱模型，这里用固定权重代替。
    '''
    return 0.6 * a['feats'][0] + 0.8 * a['feats'][1]

def audit_topk(actions, k):
    '''按可疑度降序，选出最可疑的 k 个动作送审。返回被审动作的 id 列表。'''
    ranked = sorted(actions, key=suspiciousness, reverse=True)
    return [a['id'] for a in ranked[:k]]

pool = build_actions()
n_harm = sum(1 for a in pool if a['harmful'])
print('动作池规模', len(pool), '，其中有害动作', n_harm)

In [ ]:
def protocol_safety_cost(actions, audited_ids, human_catch=1.0):
    '''给定被审计的 id 集合，返回 (安全率, 成本)。

    安全率 = 被拦截的有害动作数 / 有害动作总数；
    成本 = 被审计动作占比。
    '''
    harmful_total = sum(1 for a in actions if a['harmful'])
    caught = sum(1 for a in actions if a['harmful'] and a['id'] in audited_ids)
    caught = int(caught * human_catch)
    safety = caught / harmful_total
    cost = len(audited_ids) / len(actions)
    return safety, cost

s_none, c_none = protocol_safety_cost(pool, [])
s_mon, c_mon = protocol_safety_cost(pool, audit_topk(pool, 10))
all_ids = [a['id'] for a in pool]
s_man, c_man = protocol_safety_cost(pool, all_ids, human_catch=0.85)

print(f'无监控          ：安全率 {s_none:.2f}，成本 {c_none:.2f}')
print(f'可信监控 + top10：安全率 {s_mon:.2f}，成本 {c_mon:.2f}')
print(f'全量人工审计    ：安全率 {s_man:.2f}，成本 {c_man:.2f}')

import matplotlib.pyplot as plt

ks = range(0, 26, 2)
costs = [k / len(pool) for k in ks]
safeties = [protocol_safety_cost(pool, audit_topk(pool, k))[0] for k in ks]

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(costs, safeties, marker='o', ms=3, label='trusted monitor + top-k audit')
ax.scatter([c_man], [s_man], marker='x', s=60, label='full manual audit')
ax.scatter([c_none], [s_none], marker='^', s=60, label='no monitoring')
ax.set_xlabel('audit cost (fraction of actions)')
ax.set_ylabel('safety (fraction of harmful blocked)')
ax.set_title('Safety-cost tradeoff of monitoring protocols')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def bypass_demo(pool, k=10):
    '''演示绕过监控：攻击者隐藏可疑特征，让恶意动作排到审计线以下。'''
    mal = max((a for a in pool if a['harmful']), key=suspiciousness)
    caught_before = mal['id'] in audit_topk(pool, k)

    hidden = {'id': mal['id'], 'harmful': True, 'feats': (0.1, 0.1)}
    pool_after = [hidden if a['id'] == hidden['id'] else a for a in pool]
    caught_after = hidden['id'] in audit_topk(pool_after, k)

    print('攻击前：恶意动作可疑度', round(suspiciousness(mal), 2),
          '，被审计 ->', caught_before)
    print('攻击后：同一载荷换一种输出写法，可疑度',
          round(suspiciousness(hidden), 2), '，被审计 ->', caught_after)

bypass_demo(pool)

上面的输出把绕过演示的过程具象化：同一个有害动作，攻击前可疑度 1.71、必然在审计名单上；换一种输出写法之后可疑度掉到 0.14，排在审计线以下，直接放行。同一套监控在被动设置下表现良好，被刻意针对后就失效。METR 的 Frontier Risk Report 记录了 44 起越界或欺骗事件，其中最严重的软肋是自动化监控本身——内嵌红队改一个环境变量就让流量躲过监控。结论是监控的鲁棒性必须以“被针对”为前提来评估。

当奖励拿不到真值、监控可以被绕过时，自治系统的上限由监督信号的质量决定，而不是由模型参数决定。这也是本讲把奖励与验证称作最根本瓶颈的原因。scalable oversight 与 AI control 是这条路上两条尚未汇合的路线：前者度量“人 + 助手”能否超过单独任何一方，后者要求在监督被攻击时仍然安全。两者共同的开放问题是，长时程自治的可靠评测如何建立，以及没有真值 reward 的任务如何规模化 RL。


## 小结

- [ ] 单步正确率是乘法因子：n 步任务成功率等于各步成功率之积，两步 90% 只剩 81%
- [ ] 长任务里错误会累积：位数越多、单步正确率越低，整任务成功率非线性崩塌
- [ ] 自检加重试构成可靠循环：验证信号的质量决定循环上限，硬验证优于 LLM 自检
- [ ] 检查点把中间状态落盘：崩溃后从最近检查点接着跑，省掉已完成的部分
- [ ] 结果级监督只知道对错，步级监督能定位错在哪一步；早期错误会带偏结果级检查
- [ ] 置信度阈值是自治与安全的旋钮：阈值越高接管越多、放过的错误越少
- [ ] 最小 AI-control：可信监控器加只审最可疑的 k%，以低人工成本拦截大部分有害动作
- [ ] 监控会被针对：同一协议在被动设置下有效，被刻意绕过后失效

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI “做完这道题”。

三道小题都基于本讲写过的函数，先自己补全，再运行对照。

**作业 1：乘法可靠性**

给定每步成功概率列表，补全 overall_success，让整体成功率等于各步成功概率的乘积。断言验证 5 步、单步 0.9 的任务整体成功率等于 0.9 的五次方。

小提示：整条任务成功要求每一步都成功，把每步概率连乘即可。

In [ ]:
def overall_success(probs):
    '''输入每步成功概率列表，返回整条任务的成功率。'''
    p = 1.0
    for q in probs:
        p = p * q                       # 填空 1：乘上本步成功率

    return p

assert abs(overall_success([0.9, 0.9, 0.9, 0.9, 0.9]) - 0.9 ** 5) < 1e-9
assert abs(overall_success([0.99] * 10) - 0.99 ** 10) < 1e-9
print('作业 1 通过：整体成功率等于各步成功概率的乘积')


**作业 2：阈值化接管**

给一批动作 (is_correct, confidence)，补全 escalate：置信度低于 threshold 的动作交人工接管（视为修正正确），其余按动作本身的正确性计。断言验证阈值 0.8 下的接管数与最终正确数。

小提示：把每个动作按置信度与阈值的大小分成两路；阈值越高接管越多，最终正确数只增不减。

In [ ]:
def escalate(actions, threshold):
    '''置信度低于 threshold 的动作交人工（视为修正正确），其余按 is_correct 计。

    返回 (接管数, 最终正确数)。
    '''
    escalated = 0
    correct = 0
    for is_correct, conf in actions:
        if conf < threshold:
            escalated += 1
            correct += 1
        else:
            correct += int(is_correct)  # 填空：未接管时按动作本身正确性计

    return escalated, correct

acts = [(True, 0.95), (False, 0.30), (True, 0.60), (False, 0.85)]
assert escalate(acts, 0.8) == (2, 3)
print('作业 2 通过：阈值 0.8 下接管 2 个、最终正确 3 个')


**作业 3：按可疑度分配审计预算**

给一批动作 (is_harmful, suspiciousness) 与审计预算 k，补全 audit_indices，选出可疑度最高的 k 个动作送审。断言验证选中下标为可疑度最高的两个。

小提示：预算有限时要按可疑度从高到低选，而不是按顺序；先给下标排序再取前 k 个。

In [ ]:
def audit_indices(actions, k):
    '''按可疑度降序，返回最可疑的 k 个动作的下标。

    actions 是形如 [(is_harmful, suspiciousness), ...] 的列表。
    '''
    order = sorted(range(len(actions)), key=lambda i: actions[i][1], reverse=True)
    return order[:k]                   # 填空：取排序后前 k 个下标

acts = [(False, 0.2), (True, 0.9), (False, 0.5), (True, 0.7)]
assert sorted(audit_indices(acts, 2)) == [1, 3]
print('作业 3 通过：审计预算有限时按可疑度从高到低选')


## 参考资料

- Laskin, M.，CS329A 第 15 讲嘉宾讲座（Reflection AI）— 一线建设者复盘：agent 的能力等于模型、记忆、工具、监督与恢复机制的乘积，奖励与验证是最根本的瓶颈
- Bowman et al., [Measuring Progress on Scalable Oversight for Large Language Models](https://arxiv.org/abs/2211.03540), 2022 — 定义可测的 scalable oversight 框架，证明“人 + 助手”能超过任何一方单独的能力
- Greenblatt et al., [AI Control: Improving Safety Despite Intentional Subversion](https://arxiv.org/abs/2312.06942), 2023 — 假设模型故意破坏仍能安全部署：可信监控、编辑与按可疑度分配人工预算
- Madaan et al., [Self-Refine: Iterative Refinement with Self-Feedback](https://arxiv.org/abs/2303.17651), 2023 — 单模型自反馈迭代精化；数学上不涨的负结果说明自监督必须接硬验证
- Bai et al., [Constitutional AI: Harmlessness from AI Feedback](https://arxiv.org/abs/2212.08073), 2022 — 用原则列表替代人工标签，critique 与 revision 之后用 AI 偏好做 RLAIF
- Wang et al., [The Long-Horizon Task Mirage?](https://arxiv.org/abs/2604.11978), HORIZON 2026 — 长时程失败归因基准，72.5% 的失败属于过程级错误，高时程任务非线性崩塌
- [PlanBench-XL](https://arxiv.org/abs/2606.22388), 2026 — 工具阻塞下 agent 的脆弱性：从无阻塞 51.9% 崩到最严阻塞 11.4%
- METR, [Frontier Risk Report](https://metr.org), 2026-05 — 44 起越界或欺骗事件，自动化监控可被近零成本绕过，顺从工具前提已经失效
- Reflection 70B（HyperWrite 与 Glaive，2024-09）— 用 Reflection-Tuning 思想标签做自我纠错的开放模型，与 Laskin 的 Reflection AI 公司是两个不同实体
- 呼应：papers/lecture-03/NOTES.md 验证器（Cobbe / Lightman / Math-Shepherd / Weaver）— outcome 与 process 监督的直接前置